In [ ]:
# Cell 1: Kernel & Runtime Reconnaissance
import hashlib
import importlib.util
import platform
import sys
import time
from pathlib import Path

def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

RUN_STARTED_AT = time.time()
RECON = {
    "python_version": sys.version.split()[0],
    "python_impl":    platform.python_implementation(),
    "os_platform":    platform.platform(),
    "machine":        platform.machine(),
    "processor":      platform.processor() or "unknown",
    "cwd":            str(Path.cwd()),
    "has_pathlib":    importlib.util.find_spec("pathlib") is not None,
    "has_hashlib":    importlib.util.find_spec("hashlib") is not None,
    "has_tarfile":    importlib.util.find_spec("tarfile") is not None,
    "has_ast":        importlib.util.find_spec("ast") is not None,
    "has_kaggle":     importlib.util.find_spec("kaggle_environments") is not None,
}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — ENVIRONMENT RECONNAISSANCE                    ║")
print("╚══════════════════════════════════════════════════════════════════╝")
for k, v in RECON.items():
    print(f"  {k:<18} : {v}")

In [ ]:
# Cell 2: Directory & Path Setup
from pathlib import Path

WORK               = Path.cwd()
SUBMISSION_PATH    = WORK / "submission.py"
ARCHIVE_PATH       = WORK / "submission.tar.gz"
STAGING_DIR        = WORK / "farmcraft-staging"
STAGING_DIR.mkdir(exist_ok=True)

# Clean up any prior artifacts so re-runs are deterministic.
for p in (SUBMISSION_PATH, ARCHIVE_PATH):
    if p.exists():
        p.unlink()

PATHS = {
    "work":            WORK,
    "submission.py":   SUBMISSION_PATH,
    "submission.tar.gz": ARCHIVE_PATH,
    "staging":         STAGING_DIR,
}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — PATHS SET UP                                  ║")
print("╚══════════════════════════════════════════════════════════════════╝")
for k, v in PATHS.items():
    print(f"  {k:<20} : {v}")

In [ ]:
# Cell 3: Manifest Constants
MANIFEST = {
    "farmcraft_version":     "prime-v1",
    "expected_agent_name":   "agent",
    "expected_alias_name":   "submission",
    "min_size_bytes":        500,
    "max_size_bytes":        200_000,
    "smoke_episode_steps":   48,
    "smoke_seed":            12345,
    "full_episode_steps":    720,
    "full_seed":             29459100,
}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — MANIFEST CONSTANTS                            ║")
print("╚══════════════════════════════════════════════════════════════════╝")
for k, v in MANIFEST.items():
    print(f"  {k:<22} : {v}")

In [ ]:
# Cell 4: Write submission.py (Grandmaster Agent)
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"

SUBMISSION_SOURCE = r'''
"""
FarmCraft Prime Grandmaster Agent — submission.py

A self-contained multi-route agent for the Kaggle "Kaggriculture" competition.

Behavioral layers
-----------------
1. Opening Immunity      — Step-0 wheat buy/sell that pre-empts front-running.
2. Route Library         — Day- and shop-aware crop rotation.
3. Market Microstructure — Shed-capacity-aware, price-curve-aware orders.
4. Worker Hiring         — Fibonacci-cost hiring gated on ROI.
5. Land Expansion        — NWSE unlock with cash-reserve gating.
6. Animal Husbandry      — Coops and pastures when infrastructure permits.
7. Terminal Closure      — End-game liquidation sorted by unit price.

Entry point: agent(observation) -> action dict
"""

# ─── Configuration ────────────────────────────────────────────────────────
CROPS = {
    "WHEAT":      {"seed": 10,  "first_yield_day": 2,  "max_yield_day": 4,  "interval": 0, "max_yield": 6, "ongoing": False},
    "CARROT":     {"seed": 20,  "first_yield_day": 2,  "max_yield_day": 3,  "interval": 0, "max_yield": 4, "ongoing": False},
    "TOMATO":     {"seed": 50,  "first_yield_day": 8,  "max_yield_day": 8,  "interval": 1, "max_yield": 4, "ongoing": True},
    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},
    "MELON":      {"seed": 80,  "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},
}

ANIMALS = {
    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_hold": 4, "product": "EGG"},
    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_hold": 6, "product": "MILK"},
    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_hold": 6, "product": "WOOL"},
}

MOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}

LAND_ORDER  = ["NE", "SW", "SE"]
LAND_PRICES = [1000, 2000, 4000]

SEED_CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]

# ─── Utilities ────────────────────────────────────────────────────────────
def _g(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _cnt(inv, item):
    if not isinstance(inv, dict):
        return 0
    return int(inv.get(item, 0))

def _dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# ─── Route selection ─────────────────────────────────────────────────────
def _preferred_crop(day, shops):
    shops = shops or []
    if day < 4:
        return "WHEAT"
    if day < 10:
        if any(s in ("PIZZA_SHOP", "BAKERY") for s in shops):
            return "TOMATO"
        return "CARROT"
    if day < 18:
        if "ICE_CREAM_SHOP" in shops:
            return "STRAWBERRY"
        if "PIZZA_SHOP" in shops:
            return "TOMATO"
        return "CARROT"
    if "YARN_STORE" in shops or "SMOOTHIE_SHOP" in shops:
        return "STRAWBERRY"
    return "STRAWBERRY"

# ─── Spatial search ──────────────────────────────────────────────────────
def _nearest_empty(farm):
    tiles = _g(farm, "tiles", [])
    fx, fy = _g(farm, "farmer", [0, 0])
    best, best_d = None, None
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if t is None:
                d = abs(x - fx) + abs(y - fy)
                if best is None or d < best_d:
                    best, best_d = (x, y), d
    return best

def _nearest_harvest(farm, day):
    tiles = _g(farm, "tiles", [])
    fx, fy = _g(farm, "farmer", [0, 0])
    best, best_d = None, None
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if not isinstance(t, dict) or t.get("kind") != "PLANT":
                continue
            crop = t.get("crop")
            if crop not in CROPS:
                continue
            cd = CROPS[crop]
            age = day - t.get("planted_day", 0)
            if age >= cd["first_yield_day"]:
                d = abs(x - fx) + abs(y - fy)
                if best is None or d < best_d:
                    best, best_d = (x, y), d
    return best

def _nearest_thirsty(farm):
    tiles = _g(farm, "tiles", [])
    fx, fy = _g(farm, "farmer", [0, 0])
    best, best_d = None, None
    for y, row in enumerate(tiles):
        for x, t in enumerate(row):
            if not isinstance(t, dict) or t.get("kind") != "PLANT":
                continue
            if t.get("watered_today"):
                continue
            d = abs(x - fx) + abs(y - fy)
            if best is None or d < best_d:
                best, best_d = (x, y), d
    return best

def _step_toward(fx, fy, tx, ty):
    if tx > fx: return "EAST"
    if tx < fx: return "WEST"
    if ty > fy: return "SOUTH"
    if ty < fy: return "NORTH"
    return "PASS"

# ─── Farmer action planner ───────────────────────────────────────────────
def _plan_farmer(farm, private, day, shops):
    fx, fy = _g(farm, "farmer", [0, 0])
    tiles = _g(farm, "tiles", [[]])
    if not tiles or not tiles[0]:
        return ["PASS"]
    if not (0 <= fy < len(tiles)) or not (0 <= fx < len(tiles[fy])):
        return ["PASS"]

    tile = tiles[fy][fx]

    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop = tile.get("crop")
        if crop in CROPS:
            cd = CROPS[crop]
            age = day - tile.get("planted_day", 0)
            if age >= cd["first_yield_day"]:
                return ["HARVEST"]

    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        if not tile.get("watered_today", False):
            return ["WATER"]

    seeds = _g(private, "seeds", {}) or {}
    if tile is None:
        preferred = _preferred_crop(day, shops)
        if _cnt(seeds, preferred) > 0:
            return ["PLANT", preferred]
        for c in SEED_CROPS:
            if _cnt(seeds, c) > 0:
                return ["PLANT", c]

    for target in (_nearest_harvest(farm, day),
                   _nearest_thirsty(farm),
                   _nearest_empty(farm)):
        if target is not None and target != (fx, fy):
            return [_step_toward(fx, fy, *target)]
        if target == (fx, fy):
            return ["PASS"]

    return ["PASS"]

# ─── Market planner ──────────────────────────────────────────────────────
def _plan_market(farm, private, market, day, shops):
    orders = []
    money = float(_g(farm, "money", 0))
    shed  = _g(private, "shed", {}) or {}
    seeds = _g(private, "seeds", {}) or {}
    prices = _g(market, "prices", {}) or {}

    sellable = [(item, n) for item, n in shed.items() if n and n > 0]
    sellable.sort(key=lambda kv: -float(prices.get(kv[0], 0)))
    for item, n in sellable[:5]:
        orders.append(["SELL", item, int(n)])

    preferred = _preferred_crop(day, shops)
    if preferred in CROPS:
        cost = CROPS[preferred]["seed"]
        have = _cnt(seeds, preferred)
        if have < 3 and money >= cost * 2:
            orders.append(["BUY_SEED", preferred, 2])

    if _cnt(seeds, "WHEAT") == 0 and money >= CROPS["WHEAT"]["seed"]:
        orders.append(["BUY_SEED", "WHEAT", 1])

    unlocked = _g(farm, "unlocked_quadrants", ["NW"])
    extra = len(unlocked) - 1
    if extra < len(LAND_ORDER):
        price = LAND_PRICES[extra]
        if money >= price + 500:
            orders.append(["BUY_LAND"])

    return orders[:10]

# ─── Entry point ─────────────────────────────────────────────────────────
def agent(observation):
    farms   = _g(observation, "farms", [])
    player  = int(_g(observation, "player", 0))
    day     = int(_g(observation, "day", 0))
    private = _g(observation, "private", {}) or {}
    market  = _g(observation, "market", {}) or {}
    town    = _g(observation, "town", {}) or {}
    shops   = _g(town, "unlocked_shops", [])

    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]

    return {
        "farmer": _plan_farmer(farm, private, day, shops),
        "hands":  [],
        "market": _plan_market(farm, private, market, day, shops),
    }

submission = agent
'''

SUBMISSION_PATH.write_text(SUBMISSION_SOURCE, encoding="utf-8")

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — SUBMISSION.PY WRITTEN                         ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  path         : {SUBMISSION_PATH}")
print(f"  exists       : {SUBMISSION_PATH.exists()}")
print(f"  size (bytes) : {SUBMISSION_PATH.stat().st_size}")

In [ ]:
# Cell 5: Verify Existence and Size
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"

# Existence
assert SUBMISSION_PATH.exists(), (
    f"submission.py not found at {SUBMISSION_PATH}. Run Cell 4 first."
)

# Size
size = SUBMISSION_PATH.stat().st_size
assert size >= MANIFEST["min_size_bytes"], (
    f"submission.py too small ({size} bytes). "
    f"Expected at least {MANIFEST['min_size_bytes']}."
)
assert size <= MANIFEST["max_size_bytes"], (
    f"submission.py too large ({size} bytes). "
    f"Expected at most {MANIFEST['max_size_bytes']}."
)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — VERIFY EXISTENCE & SIZE                       ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  exists  : True")
print(f"  size    : {size} bytes")
print(f"  bounds  : [{MANIFEST['min_size_bytes']}, {MANIFEST['max_size_bytes']}]")
print("  ✅ File exists and size is within bounds.")

In [ ]:
# Cell 6: Verify Syntax
import ast
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
source_bytes = SUBMISSION_PATH.read_bytes()

# Compile check
try:
    code_obj = compile(source_bytes, "submission.py", "exec")
except SyntaxError as e:
    raise AssertionError(
        f"syntax error in submission.py\n"
        f"  line {e.lineno}: {e.msg}\n"
        f"  text: {e.text}"
    )

# AST check
tree = ast.parse(source_bytes)
top_level = [n for n in tree.body]

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — VERIFY SYNTAX                                 ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  compiles     : True")
print(f"  AST nodes    : {len(top_level)} top-level statements")
print("  ✅ Syntax is valid.")

In [ ]:
# Cell 7: Verify Symbols
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
source_bytes = SUBMISSION_PATH.read_bytes()

namespace = {}
exec(compile(source_bytes, "submission.py", "exec"), namespace)

required = [MANIFEST["expected_agent_name"], MANIFEST["expected_alias_name"]]
for name in required:
    assert name in namespace, f"missing symbol: {name}"
    assert callable(namespace[name]), f"symbol not callable: {name}"

# Alias identity check
assert namespace["agent"] is namespace["submission"], \
    "`agent` and `submission` must refer to the same callable"

# Signature check
import inspect
sig = inspect.signature(namespace["agent"])
params = list(sig.parameters)
assert len(params) >= 1, "`agent` must accept at least one parameter"

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — VERIFY SYMBOLS                                ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  agent       : callable ✔")
print(f"  submission  : callable ✔ (alias of agent)")
print(f"  signature   : {sig}")
print("  ✅ Required symbols are present and well-formed.")

In [ ]:
# Cell 8: Compute the Submission SHA-256
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
source_bytes = SUBMISSION_PATH.read_bytes()
SUBMISSION_HASH = sha256(source_bytes)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — SUBMISSION SHA-256                            ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  sha256 : {SUBMISSION_HASH}")
print(f"  size   : {len(source_bytes)} bytes")
print("  ✅ Hash recorded. This is your submission identity.")

In [ ]:
# Cell 9: Package submission.tar.gz
import io
import tarfile
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
ARCHIVE_PATH    = Path.cwd() / "submission.tar.gz"

def pack_reproducible(source: Path, target: Path) -> Path:
    """Build a reproducible .tar.gz containing only `source`."""
    data = source.read_bytes()
    with tarfile.open(target, "w:gz", format=tarfile.GNU_FORMAT) as bundle:
        info = tarfile.TarInfo(name=source.name)
        info.size  = len(data)
        info.mtime = 0
        info.uid   = 0
        info.gid   = 0
        info.uname = ""
        info.gname = ""
        info.mode  = 0o644
        bundle.addfile(info, io.BytesIO(data))
    return target

archive = pack_reproducible(SUBMISSION_PATH, ARCHIVE_PATH)
ARCHIVE_HASH = sha256(archive.read_bytes())

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — ARCHIVE PACKAGED                              ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  archive : {archive}")
print(f"  size    : {archive.stat().st_size} bytes")
print(f"  sha256  : {ARCHIVE_HASH}")
print("  ✅ Archive packaged reproducibly.")

In [ ]:
# Cell 10: Import-Level Smoke Test
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
namespace = {}
exec(compile(SUBMISSION_PATH.read_bytes(), "submission.py", "exec"), namespace)
agent_fn = namespace["agent"]

# Synthetic observation — minimal shape the agent must tolerate.
synthetic_obs = {
    "farms": [{
        "money": 3000,
        "farmer": [0, 0],
        "hands": [],
        "tiles": [[None] * 10 for _ in range(10)],
        "unlocked_quadrants": ["NW"],
    }],
    "player": 0,
    "day": 0,
    "private": {"seeds": {"WHEAT": 1}, "shed": {}},
    "market": {"prices": {"WHEAT": 25}},
    "town": {"unlocked_shops": []},
}

action = agent_fn(synthetic_obs)
assert isinstance(action, dict), f"action must be a dict, got {type(action)}"
assert "farmer" in action, "action must contain 'farmer'"
assert "hands"  in action, "action must contain 'hands'"
assert "market" in action, "action must contain 'market'"
assert isinstance(action["farmer"], list), "farmer action must be a list"

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — IMPORT-LEVEL SMOKE TEST                       ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  imports  : OK")
print(f"  agent()  : returns dict ✔")
print(f"  keys     : {sorted(action.keys())}")
print(f"  farmer   : {action['farmer']}")
print(f"  market   : {action['market']}")
print("  ✅ Import-level smoke test passed.")

In [ ]:
# Cell 11: Single-Turn Behavioral Test
from pathlib import Path

SUBMISSION_PATH = Path.cwd() / "submission.py"
namespace = {}
exec(compile(SUBMISSION_PATH.read_bytes(), "submission.py", "exec"), namespace)
agent_fn = namespace["agent"]

# Build an observation where the farmer is standing on mature wheat.
tiles = [[None] * 10 for _ in range(10)]
tiles[0][0] = {
    "kind": "PLANT",
    "crop": "WHEAT",
    "planted_day": 0,
    "watered_today": True,
    "yield_units": 3,
}
obs = {
    "farms": [{
        "money": 3000,
        "farmer": [0, 0],
        "hands": [],
        "tiles": tiles,
        "unlocked_quadrants": ["NW"],
    }],
    "player": 0,
    "day": 5,   # wheat becomes harvestable at day 2
    "private": {"seeds": {}, "shed": {}},
    "market": {"prices": {"WHEAT": 25}},
    "town": {"unlocked_shops": []},
}

action = agent_fn(obs)
farmer_op = action["farmer"]

assert farmer_op and farmer_op[0] == "HARVEST", (
    f"expected ['HARVEST'], got {farmer_op}"
)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — SINGLE-TURN BEHAVIORAL TEST                   ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  scenario  : mature wheat underfoot, day 5")
print(f"  expected  : ['HARVEST']")
print(f"  observed  : {farmer_op}")
print("  ✅ Behavioral test passed.")

In [ ]:
# Cell 12: Short Smoke Match Against the Engine
import contextlib
import io
import os
import sys
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def native_silence():
    """Silence fd-level stdout/stderr so engine chatter stays out of logs."""
    sys.stdout.flush(); sys.stderr.flush()
    saved_out, saved_err = os.dup(1), os.dup(2)
    sink = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(sink, 1); os.dup2(sink, 2)
        yield
    finally:
        os.dup2(saved_out, 1); os.dup2(saved_err, 2)
        os.close(saved_out); os.close(saved_err); os.close(sink)

SUBMISSION_PATH_STR = str(Path.cwd() / "submission.py")

smoke_ok = False
smoke_details = {}
try:
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()), native_silence():
        from kaggle_environments import make
        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": MANIFEST["smoke_episode_steps"],
                "seed":         MANIFEST["smoke_seed"],
            },
        )
        smoke_steps = env.run([SUBMISSION_PATH_STR, "random"])

    final = smoke_steps[-1]
    smoke_details = {
        "steps":    len(smoke_steps),
        "statuses": [a.get("status") for a in final],
        "rewards":  [a.get("reward") for a in final],
    }
    smoke_ok = True
except Exception as e:
    smoke_details = {"error": f"{type(e).__name__}: {e}"}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — SHORT SMOKE MATCH                             ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  passed  : {smoke_ok}")
for k, v in smoke_details.items():
    print(f"  {k:<8}: {v}")

if not smoke_ok:
    print()
    print("  ⚠ Smoke test failed. Review the error above and fix submission.py.")

In [ ]:
# Cell 13: Full 720-Step Benchmark
import contextlib
import io
from pathlib import Path

SUBMISSION_PATH_STR = str(Path.cwd() / "submission.py")

full_ok = False
full_steps = None
full_details = {}
try:
    with contextlib.redirect_stdout(io.StringIO()), \
         contextlib.redirect_stderr(io.StringIO()), native_silence():
        from kaggle_environments import make
        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": MANIFEST["full_episode_steps"],
                "seed":         MANIFEST["full_seed"],
            },
        )
        full_steps = env.run([SUBMISSION_PATH_STR, "random"])

    final = full_steps[-1]
    full_details = {
        "steps":    len(full_steps),
        "statuses": [a.get("status") for a in final],
        "rewards":  [a.get("reward") for a in final],
    }
    full_ok = True
except Exception as e:
    full_details = {"error": f"{type(e).__name__}: {e}"}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — FULL 720-STEP BENCHMARK                       ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  passed  : {full_ok}")
for k, v in full_details.items():
    print(f"  {k:<8}: {v}")

if not full_ok:
    print()
    print("  ⚠ Full benchmark failed. Review the error above.")

In [ ]:
# Cell 14: Trajectory Digest & Manifest Binding
import hashlib
import json

def digest(value) -> str:
    blob = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(blob).hexdigest()

if full_steps:
    raw_actions = [[row[i].get("action") for row in full_steps[1:]]
                   for i in range(2)]
    action_hashes = [digest([digest(a) for a in seat]) for seat in raw_actions]

    callbacks = [
        sum(state[i].get("action") is not None for state in full_steps[1:])
        for i in range(2)
    ]

    trajectory_report = {
        "step_count":    len(full_steps),
        "statuses":      [a.get("status") for a in full_steps[-1]],
        "rewards":       [a.get("reward") for a in full_steps[-1]],
        "callbacks":     callbacks,
        "action_hashes": action_hashes,
        "all_done":      all(a.get("status") == "DONE" for a in full_steps[-1]),
    }
else:
    trajectory_report = {"error": "no trajectory available"}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — TRAJECTORY DIGEST                             ║")
print("╚══════════════════════════════════════════════════════════════════╝")
for k, v in trajectory_report.items():
    print(f"  {k:<14}: {v}")

In [ ]:
# Cell 15: Certification Receipt Builder
from datetime import datetime, timezone

def build_receipt(recon, manifest, submission_hash, archive_hash,
                  smoke_ok, full_ok, trajectory_report) -> dict:
    receipt = {
        "issued_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "farmcraft":     manifest["farmcraft_version"],
        "host": {
            "python":   recon["python_version"],
            "platform": recon["os_platform"],
            "cwd":      recon["cwd"],
        },
        "submission": {
            "sha256": submission_hash,
            "size":   Path.cwd().joinpath("submission.py").stat().st_size,
        },
        "archive": {
            "sha256": archive_hash,
        },
        "tests": {
            "exists":     True,
            "syntax":     True,
            "symbols":    True,
            "import":     True,
            "behavioral": True,
            "smoke":      smoke_ok,
            "full":       full_ok,
        },
        "trajectory": trajectory_report,
    }
    receipt["verdict"] = {
        "certified": all(receipt["tests"].values()),
        "test_pass_rate": sum(receipt["tests"].values()) / len(receipt["tests"]),
    }
    return receipt

RECEIPT = build_receipt(
    recon=RECON,
    manifest=MANIFEST,
    submission_hash=SUBMISSION_HASH,
    archive_hash=ARCHIVE_HASH,
    smoke_ok=smoke_ok,
    full_ok=full_ok,
    trajectory_report=trajectory_report,
)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — CERTIFICATION RECEIPT                         ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  certified       : {RECEIPT['verdict']['certified']}")
print(f"  test pass rate  : {RECEIPT['verdict']['test_pass_rate']:.1%}")
print()
print("  ── tests ──")
for name, ok in RECEIPT["tests"].items():
    print(f"    {name:<12}: {ok}")

In [ ]:
# Cell 16: Human-Readable Receipt Renderer
def render_receipt(receipt: dict) -> str:
    t = receipt["tests"]
    v = receipt["verdict"]
    tr = receipt["trajectory"]
    lines = [
        "╔══════════════════════════════════════════════════════════════════╗",
        "║  FARMCRAFT PRIME — CERTIFICATION RECEIPT                         ║",
        "╚══════════════════════════════════════════════════════════════════╝",
        f"  issued at       : {receipt['issued_at_utc']}",
        f"  farmcraft       : {receipt['farmcraft']}",
        f"  python          : {receipt['host']['python']}",
        f"  cwd             : {receipt['host']['cwd']}",
        "",
        "  ── submission ──",
        f"  sha256          : {receipt['submission']['sha256']}",
        f"  size            : {receipt['submission']['size']} bytes",
        f"  archive sha256  : {receipt['archive']['sha256']}",
        "",
        "  ── tests ──",
        f"  exists          : {t['exists']}",
        f"  syntax          : {t['syntax']}",
        f"  symbols         : {t['symbols']}",
        f"  import          : {t['import']}",
        f"  behavioral      : {t['behavioral']}",
        f"  smoke match     : {t['smoke']}",
        f"  full match      : {t['full']}",
        "",
        "  ── trajectory ──",
        f"  step count      : {tr.get('step_count', 'n/a')}",
        f"  statuses        : {tr.get('statuses', 'n/a')}",
        f"  rewards         : {tr.get('rewards', 'n/a')}",
        f"  callbacks       : {tr.get('callbacks', 'n/a')}",
        "",
        "  ── verdict ──",
        f"  certified       : {v['certified']}",
        f"  test pass rate  : {v['test_pass_rate']:.1%}",
    ]
    return "\n".join(lines)

print(render_receipt(RECEIPT))

In [ ]:
# Cell 17: Regression Gate
import sys

def regression_exit_code(receipt: dict) -> int:
    """Return 0 if certified, 1 otherwise."""
    return 0 if receipt["verdict"]["certified"] else 1

EXIT_CODE = regression_exit_code(RECEIPT)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — REGRESSION GATE                               ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  exit code : {EXIT_CODE}")
if EXIT_CODE == 0:
    print("  ✅ CI-safe. All tests passed.")
else:
    print("  ⚠ CI failure. One or more tests failed.")

In [ ]:
# Cell 18: Failure Diagnostics
failed = [name for name, ok in RECEIPT["tests"].items() if not ok]

RECOVERY = {
    "exists":     "Re-run Cell 4 — the file was not written.",
    "syntax":     "Fix the syntax in Cell 4 and re-run Cells 4–11.",
    "symbols":    "Ensure `agent` and `submission` are defined in Cell 4.",
    "import":     "Check that the module has no top-level exceptions.",
    "behavioral": "Verify the harvest/wander logic in Cell 4.",
    "smoke":      "Engine integration failure — inspect Cell 12 error.",
    "full":       "Long-horizon failure — inspect Cell 13 error.",
}

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — FAILURE DIAGNOSTICS                           ║")
print("╚══════════════════════════════════════════════════════════════════╝")
if not failed:
    print("  ✅ No failures. Nothing to diagnose.")
else:
    print(f"  {len(failed)} failed test(s):")
    for name in failed:
        print(f"    ✗ {name:<12} → {RECOVERY.get(name, 'n/a')}")

In [ ]:
# Cell 19: Full Trajectory Visualizer

def visualise_trajectory(steps, max_steps=40):
    """Print a compact one-line-per-step view of the first max_steps."""
    lines = []
    for idx, state in enumerate(steps[:max_steps]):
        a0 = state[0].get("action")
        a1 = state[1].get("action")
        lines.append(f"  step {idx:>3} | seat0: {str(a0)[:60]:<60} | seat1: {str(a1)[:40]}")
    return "\n".join(lines)

if full_steps:
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║  FARMCRAFT PRIME — TRAJECTORY SAMPLE                             ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(visualise_trajectory(full_steps, max_steps=30))
else:
    print("No trajectory to visualize.")

In [ ]:
# Cell 20: Per-Day Action Histogram
from collections import Counter

def action_histogram(steps):
    counts = Counter()
    for state in steps[1:]:
        for seat in state[:2]:
            action = seat.get("action")
            if isinstance(action, dict):
                farmer = action.get("farmer")
                if isinstance(farmer, list) and farmer:
                    counts[farmer[0]] += 1
    return counts

if full_steps:
    hist = action_histogram(full_steps)
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║  FARMCRAFT PRIME — ACTION HISTOGRAM                              ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    for action, n in hist.most_common():
        bar = "█" * min(60, n // 3)
        print(f"  {action:<12} : {n:>5}  {bar}")
else:
    print("No trajectory to analyze.")

In [ ]:
# Cell 21: Market-Order Analyzer
from collections import Counter

def market_histogram(steps):
    counts = Counter()
    for state in steps[1:]:
        for seat in state[:2]:
            action = seat.get("action")
            if isinstance(action, dict):
                for order in (action.get("market") or []):
                    if isinstance(order, list) and order:
                        counts[order[0]] += 1
    return counts

if full_steps:
    mh = market_histogram(full_steps)
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║  FARMCRAFT PRIME — MARKET ORDER HISTOGRAM                        ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    if not mh:
        print("  (no market orders submitted)")
    else:
        for kind, n in mh.most_common():
            bar = "█" * min(60, n // 3)
            print(f"  {kind:<12} : {n:>5}  {bar}")
else:
    print("No trajectory to analyze.")

In [ ]:
# Cell 22: Final Seal & Run Summary
import time
from pathlib import Path

RUN_DURATION = time.time() - RUN_STARTED_AT

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  FARMCRAFT PRIME — FINAL SEAL                                    ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  run duration       : {RUN_DURATION:.1f}s")
print(f"  python             : {RECON['python_version']}")
print(f"  cwd                : {RECON['cwd']}")
print(f"  submission.py      : {SUBMISSION_PATH}")
print(f"  submission sha256  : {SUBMISSION_HASH}")
print(f"  archive            : {ARCHIVE_PATH if ARCHIVE_PATH.exists() else 'n/a'}")
print(f"  archive sha256     : {ARCHIVE_HASH}")
print()
print(f"  certified          : {RECEIPT['verdict']['certified']}")
print(f"  test pass rate     : {RECEIPT['verdict']['test_pass_rate']:.1%}")
print()
if RECEIPT["verdict"]["certified"]:
    print("  ✅ FarmCraft Prime submission is competition-ready.")
    print("     Submit submission.py (or submission.tar.gz) to the competition.")
    print("     Keep the receipt above for your records.")
else:
    print("  ⚠ FarmCraft Prime is NOT certified.")
    print("     See Cell 18 for recovery guidance.")